In [2]:
import sys
!{sys.executable} -m pip install pandas scikit-learn

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached numpy-2.4.6-cp314-cp314-win_amd64.whl (12.5 MB)
Using cached scipy-1.17.1-cp314-cp314-win_amd64.whl (37.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   ---------------------------------------- 0/7 [tzdata]
   ----------- -------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
wine = pd.read_csv("red_wine_quality.csv")

In [8]:
data = wine[["alcohol", "sugar", "pH"]]
target = wine["class"]

In [10]:
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target,test_size=0.2, random_state=42)

In [12]:
# 그 다음 갑자기 train_input과 train_target을 다시 train_test_split 함수에 넣어서 
# 훈련세트 sub_input, sub_target과 검증세트 val_input, val_target을 만듭니다.
# 여기에서도 test_size 매개변수는 0.2로 지정해서 테스트는 20%만큼 val_input으로 만듭니다.
# 왜 그럴까 생각하며 찾아보니, 

지금 데이터가 이렇게 나뉘어 있습니다
전체 데이터
├── train (훈련용 80%)
└── test (시험용 20%)  ← 절대 안 건드림

문제 상황
모델을 만들면서 이것저것 바꿔봐야 함:
max_depth 3? 5? 7?
어떤 게 좋은지 점수를 봐야 함

근데 test로 점수 보면?
→ test를 자꾸 보게 됨
→ test에 맞춰서 모델을 튜닝하게 됨
→ test가 더 이상 "처음 보는 데이터"가 아님
→ 시험지 미리 본 셈

그래서 train을 또 쪼갭니다
train (80%)
├── sub_input (진짜 훈련용)
└── val_input (검증용) ← 튜닝할 때 여기로 점수 봄

test (20%)는 마지막까지 봉인 ← 진짜 최종 시험

비유
시험 공부할 때:

교과서 = 훈련 (sub)
모의고사 = 검증 (val) ← 실력 점검용, 여러 번 봄
수능 = 테스트 (test) ← 딱 한 번, 진짜 실력
모의고사로 부족한 부분 파악하고 보완합니다. 수능은 마지막에 딱 한 번만 봅니다.

정리
sub_input  → 모델 학습
val_input  → 어떤 설정이 좋은지 점검 (여러 번)
test_input → 최종 성능 (딱 한 번, 봉인)

그런데 한 가지
검증 세트를 직접 나누는 건 교차 검증(cross_validate)을 이해하기 위한 사전 단계입니다. 실제로는 이렇게 발전합니다.
검증 세트 1개 → 운에 좌우됨
        ↓
교차 검증 (K-Fold) → 여러 번 나눠 평균 → 더 신뢰
어제 배운 cross_validate가 이 검증 세트를 5번 자동으로 만들어주는 겁니다.

이제 이해했습니다.

In [14]:
from sklearn.model_selection import train_test_split
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target,test_size=0.2, random_state=42)

그렇죠. 이렇게 train_test_split 함수를 2번 해서 훈련세트와 검증세트로 나누는 거군요. 이제 이해가 됩니다.

In [15]:
#  훈련세트와 검증세트 크기 비교
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


# 줄은 숫자를 확인할 수 있습니다.
# 이제 검증세트를 가지고 모델을 만들고 평가해보겠습니다.

In [17]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.8634615384615385


오호~~~ 과대적합이네여. 이를 해결하기 위해 매개변수를 수정할려는데 책님은 갑자기 검증세트에 관해 더 알려준다고 합니다,,,,,,,,

교차검증
우리는 위에서 검증세트를 만든다고 훈련세트의 20%를 검증세트로 사용했는데요. 이렇게 되니까 훈련세트가 줄어서 훈련데이터가 부족하다고 합니다..
책님은 이때는 교차검증이라는 이용하면 안정적인 검증 점수를 얻고 훈련에 더 많은 데이터를 사용할 수 있다고 말해줍니다. 왜 지금에서야 말해줄까여,,

교차검증은 검증세트를 떼어내어 평가하는 과정을 여러번 반복합니다. 그 다음 이 점수를 평균내어 최종검증점수를 얻습니다. 이를 폴드 교차검증이라고 하는데요. 폴드 교차검증은 훈련세트를 몇번 나누냐에 따라 제목이 정해진다고 하네용. 그냥 훈련세트 사이에 검증세트의 순서를 바꿔가며 검증하는 걸 말합니다.


그런데 보통은 5차나 10차 폴드 교차검증을 한다고 하네요. 이렇게 하면 데이터의 80~90%를 훈련에 사용 가능해서 검증세트가 줄어들지만 각 폴드애서 계산한 검증 점수를 평균하기 때문에 안정된 점수로 생각할 수 있습니다.

그런데 사이킷런에는 cross_validate라는 교차 검증 함수가 있다고 합니다. 사용법은 먼저 평가할 모델 객체를 첫번째 매개변수로 전달하고, 그 다음 앞에서처럼 직접 검증세트를 떼어내지 않고 훈련세트전체를 cross_validate()함수에 전달.

In [18]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.00616574, 0.00534821, 0.00561237, 0.00548005, 0.005651  ]), 'score_time': array([0.00171041, 0.0015738 , 0.00168562, 0.00153542, 0.00284767]), 'test_score': array([0.86634615, 0.85480769, 0.88161694, 0.84985563, 0.84119346])}


이 함수는 fit_time, score_time, test_score 키를 가진 딕셔너리를 반환합니다.
처음 2개는 각각 모델을 훈련하는 시간과 검증하는 시간을 의미하며, 각 키마다 5개의 숫자 포함됨.
cross_validate()는 기본적으로 5폴드 교차검증을 수행.
cv매개변수에서 폴드 수를 변경 가능
교차검증의 최종 점수는 test_score키에 담긴 5개의 점수를 평균하여 얻을 수 있습니다. 

In [20]:
import numpy as np
print(np.mean(scores["test_score"]))

0.8587639742355815


교차검증을 통해 입력한 모델에서 얻을 수 있는 최상의 검증 점수를 가늠할 수 있습니다.
한 가지 주의할 점은 cross_validate는 훈련 세트를 섞어 폴드를 나누지 않기에 앞서 train_test_split을 통해 했기에 따로 안 섞어도 됩니다. 
다만!!, 교차검증을 할 때 훈련세트를 섞으려면 분할기를 지정해줘야 합니다.

사이킷런의 분할기는 교차 검증에서 폴드를 어떻게 나눌지 결정해준다고 합니다. cross_validate 함수는 기본적으로 회귀 모델인 경우, kFold 분할기를 사용하고 분류 모델인 경우, 타깃 클래스를 골고루 나누기 위해 StratifiedKFold를 사용합니다.  

왜 섞냐면

안 섞으면 생기는 문제
데이터가 정렬되어 있을 때가 위험합니다.
와인 데이터가 이렇게 정렬돼 있다고 치면:

앞부분: 레드 레드 레드 ... (0)
뒷부분: 화이트 화이트 화이트 ... (1)
이걸 안 섞고 5조각으로 나누면:
1조각: 전부 레드
2조각: 전부 레드
3조각: 레드+화이트
4조각: 전부 화이트
5조각: 전부 화이트
1조각으로 검증하면 레드만 있어서 화이트를 한 번도 못 봅니다. 검증이 엉망이 됩니다.

섞으면
레드 화이트 레드 화이트 레드 ... (골고루)
→ 5조각 어디를 봐도 레드/화이트 섞여있음
→ 공정한 검증

비유
반 학생을 키 순서로 세워놓고
앞에서 5명씩 조를 짜면:
1조 = 키 작은 애들만
5조 = 키 큰 애들만
→ 불공평

섞어서 조를 짜면:
모든 조에 큰 애 작은 애 골고루
→ 공정

그런데 왜 "안 섞어도 된다"고 했냐면
이미 train_test_split이 섞었음!

train_test_split은 데이터를 무작위로 섞어서 나눔
→ train 자체가 이미 골고루 섞인 상태
→ cross_validate가 또 안 섞어도 괜찮음

그럼 언제 분할기로 섞냐면
혹시 모를 안전장치 + 더 확실하게 하고 싶을 때
→ StratifiedKFold 같은 분할기 지정
→ 클래스 비율까지 맞춰서 섞음
pythonfrom sklearn.model_selection import StratifiedKFold

splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(model, train_input, train_target, cv=splitter)
옵션의미n_splits=55조각shuffle=True섞기StratifiedKFold클래스 비율 유지하며 섞기

정리
섞는 이유: 데이터가 정렬돼 있으면 검증이 편향됨
이미 섞임: train_test_split이 섞어놨으니 보통은 OK
분할기 사용: 더 확실하게 + 클래스 비율까지 맞추고 싶을 때

In [21]:
from sklearn.model_selection import StratifiedKFold
scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.8566474790849188


scores['test_score']

scores가 뭐냐면
cross_validate가 반환하는 딕셔너리(여러 정보 묶음) 입니다.
pythonscores = {
    'fit_time':    [0.21, 0.21, 0.22, 0.21, 0.21],   # 학습 시간
    'score_time':  [0.01, 0.01, 0.01, 0.01, 0.01],   # 평가 시간
    'test_score':  [0.85, 0.87, 0.84, 0.86, 0.88]    # ← 검증 점수!
}

scores['test_score']
딕셔너리에서 'test_score' 열쇠로 검증 점수만 꺼내는 것입니다.
pythonscores['test_score']
# → [0.85, 0.87, 0.84, 0.86, 0.88]
#    5폴드 각각의 검증 점수

비유
scores = 성적표 전체
'test_score' = 성적표에서 "검증 점수" 칸만 콕 집기

사물함(scores)에서
'test_score'라는 이름표 붙은 칸 열기
→ 5개 점수 꺼냄

전체 코드 흐름
pythonscores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
# → 5폴드 결과 전체 (딕셔너리)

scores['test_score']
# → 5개 검증 점수만 추출 [0.85, 0.87, ...]

np.mean(scores['test_score'])
# → 5개 평균 = 0.86 (진짜 실력, 수렴값)

왜 평균을 내냐면
[0.85, 0.87, 0.84, 0.86, 0.88]  ← 5개 점수
→ 하나만 보면 운에 좌우됨
→ 평균 0.86 = 안정적인 진짜 실력

주의 — 'train_score'는 왜 없냐면
cross_validate 기본값은 test_score만 반환
훈련 점수도 보려면:
cross_validate(..., return_train_score=True)
→ 그러면 scores['train_score']도 생김

만약, 훈련세트를 섞은 후 10폴드 교차 검증을 수행할려면 다음과 같이 작성.

In [22]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt,train_input, train_target, cv=splitter)
print(np.mean(scores["test_score"]))

0.85953683118423


이제 우리의 주인공은 이어서 결정트리의 매개변수 값을 바꿔가며 가장 좋은 성능이 나오는 모델을 찾아본다고 합니다.
이제 테스트 세트를 사용하지 않고 교차 검증을 통해서 좋은 모델을 고를 수 있다 하네요. 힘들댜~

그리고 위의 방법을 하이퍼파라미터 튜닝이라고 합니다!